In [0]:
dbutils.library.restartPython()

In [0]:
import os
os.environ['AWS_ACCESS_KEY_ID'] = dbutils.secrets.get('aws-credentials', 'AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = dbutils.secrets.get('aws-credentials', 'AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'

In [0]:
import boto3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
# URL del workspace
workspace_url = spark.conf.get("spark.databricks.workspaceUrl", "No disponible")
print(f"Workspace URL: {workspace_url}")

# Obtener región desde el session implícito de boto3
session = boto3.session.Session()
region = session.region_name # or "us-east-1" 

print(f"Región AWS detectada: {region}")

env_region = os.environ.get('AWS_REGION', os.environ.get('AWS_DEFAULT_REGION', 'No configurada'))
print(f"Región desde variables de entorno: {env_region}")

In [0]:
import pandas as pd
ruta = "s3://ml-models-sasa/bank+marketing/bank/bank-additional-full.csv"
df_original = pd.read_csv(ruta, delimiter=";")
df_original.head(5)

In [0]:
df = df_original.copy()

In [0]:
print("df exploration")
print(df.columns)
print("==Shape==="*10)
print(df.shape)
print("==Nan==="*10)
print(df.isna().sum())
print("==Duplication==="*10)
print(df.duplicated().sum())


In [0]:
df.drop_duplicates(inplace=True)
df.duplicated().sum()

In [0]:
# Exploring each feature its type so we can dive into the best practices

cols_object = []
cols_num = []

for col in df.columns:
    if df[col].dtype == "object":
        cols_object.append(col)
    elif df[col].dtype in ["int64", "float64"]:
        cols_num.append(col)

print(f"col objects = {cols_object}")
print(f"col numeric = {cols_num}")

In [0]:
df[cols_object].head(5)

In [0]:
for col in cols_object:
    print(f"\n{col}:")
    print(f"categorias unicas: {df[col].nunique()}")
    print(f"Categoricasl con <1 %: {(df[col].value_counts(normalize=True) < 0.01).sum()}")
          

In [0]:
df['y'].value_counts(normalize=False)

In [0]:
# Tasa de conversión por categoría con hue
for col in cols_object:
    if col != 'y':
        print(f"\n{col}:")
        
        conversion = df.groupby(col)['y'].value_counts(normalize=True).unstack()
        print(conversion.sort_values('yes', ascending=False))
        
        plt.figure(figsize=(10, 6))
        
        # Contar frecuencias absolutas para el gráfico
        plot_data = df.groupby([col, 'y']).size().reset_index(name='count')
        
        sns.barplot(data=plot_data, y=col, x='count', hue='y', 
                    order=conversion.sort_values('yes', ascending=False).index)
        
        plt.title(f'Distribución de respuestas por {col}')
        plt.xlabel('Cantidad')
        plt.legend(title='Respuesta', loc='best')
        plt.tight_layout()
        plt.show()

In [0]:
for i in df[cols_object].columns:
    plt.Figure(figsize=(12,10))
    counts = df[i].value_counts(normalize=True,ascending=False)
    print(counts)
    plt.bar(counts.index, counts.values)
    plt.title(i)
    plt.xticks(rotation=90)
    plt.show()

In [0]:
for i in df[cols_object].columns:
    plt.Figure(figsize=(12,10))
    counts = df[i].value_counts(normalize=True,ascending=False)
    print(counts)
    plt.bar(counts.index, counts.values)
    plt.title(i)
    plt.xticks(rotation=90)
    plt.show()

In [0]:
df.education.value_counts()



In [0]:
# Categorical columns and its best encoding
#- education: Ordinal encoding
#- default: one hot encoding
#- housing: Ordinal encoding - Unknown variable dropped
#- loan: Ordinal encoding - Unknown variable dropped
#- contact: Ordinal encoding (since there are only 2 variables)
#- month: Ordinal encoding 
#- poutcome: One hot encoding
#- day_of_week: Ordinal encoding
#- y: Ordinal encoding

df = df[df['default']!="unknown"]

# Tratar 'unknown' como punto medio o imputar por moda
education_mapping = {
    'illiterate': 1,
    'basic.4y': 2,
    'basic.6y': 3,
    'basic.9y': 4,
    'high.school': 5,
    'professional.course': 6,
    'university.degree': 7,
    'unknown': 0  # Asignar educación media/básica
}
df['education'] = df['education'].map(education_mapping)

# Tratar 'unknown' como una tercera categoría
df['default'] = df['default'].map({'no': 0, 'unknown': 1})
df['housing'] = df['housing'].map({'no': 0, 'yes': 1, 'unknown': 2})
df['loan'] = df['loan'].map({'no': 0, 'yes': 1, 'unknown': 2})
df['day_of_week'] = df['day_of_week'].map({'mon': 1, 'tue': 2, 'wed': 3, 'thu': 4, 'fri': 5,'sat':6,'sun':7})
df['month'] = df['month'].map({'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6, 'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12})
df['y'] = df['y'].map({'no': 0, 'yes': 1})

# job - nominal sin orden → OneHot o Label encoding
job_mapping = {
    'admin.': 0, 'blue-collar': 1, 'technician': 2, 'services': 3,
    'management': 4, 'retired': 5, 'entrepreneur': 6, 'self-employed': 7,
    'unemployed': 8, 'student': 9, 'housemaid': 10, 'unknown': 11
}
df['job'] = df['job'].map(job_mapping)

# marital - nominal sin orden
df['marital'] = df['marital'].map({'married': 0, 'single': 1, 'divorced': 2, 'unknown': 3})

# contact - binaria
df['contact'] = df['contact'].map({'cellular': 0, 'telephone': 1})

# poutcome - nominal sin orden
df['poutcome'] = df['poutcome'].map({'nonexistent': 0, 'failure': 1, 'success': 2})

# job - nominal sin orden → OneHot o Label encoding
job_mapping = {
    'admin.': 0, 'blue-collar': 1, 'technician': 2, 'services': 3,
    'management': 4, 'retired': 5, 'entrepreneur': 6, 'self-employed': 7,
    'unemployed': 8, 'student': 9, 'housemaid': 10, 'unknown': 11
}
df['job'] = df['job'].map(job_mapping)

# marital - nominal sin orden
df['marital'] = df['marital'].map({'married': 0, 'single': 1, 'divorced': 2, 'unknown': 3})

# contact - binaria
df['contact'] = df['contact'].map({'cellular': 0, 'telephone': 1})

# poutcome - nominal sin orden
df['poutcome'] = df['poutcome'].map({'nonexistent': 0, 'failure': 1, 'success': 2})

df.head(5)

In [0]:
# job - nominal sin orden → OneHot o Label encoding
job_mapping = {
    'admin.': 0, 'blue-collar': 1, 'technician': 2, 'services': 3,
    'management': 4, 'retired': 5, 'entrepreneur': 6, 'self-employed': 7,
    'unemployed': 8, 'student': 9, 'housemaid': 10, 'unknown': 11
}
df['job'] = df['job'].map(job_mapping)

# marital - nominal sin orden
df['marital'] = df['marital'].map({'married': 0, 'single': 1, 'divorced': 2, 'unknown': 3})

# contact - binaria
df['contact'] = df['contact'].map({'cellular': 0, 'telephone': 1})

# poutcome - nominal sin orden
df['poutcome'] = df['poutcome'].map({'nonexistent': 0, 'failure': 1, 'success': 2})

# Numerical features

In [0]:
df[cols_num].describe()

In [0]:
for col in cols_num:
    plt.boxplot(df[col])
    plt.title(col)
    plt.show()


In [0]:
# Variables and its way to be treated

# duration:Winson's method min and max 
# campaign: Winsor's method max
# pdays: Winsor's method min 
# previous: Winsor's method max

from scipy.stats.mstats import winsorize

# Winsorizar ambos extremos (por defecto 5% inferior y 5% superior)
df['duration'] = winsorize(df['duration'], limits=[0.05, 0.05])

# Solo winsorizar el máximo (superior)
df['campaign'] = winsorize(df['campaign'], limits=[0, 0.05])  # 0% abajo, 5% arriba

# Solo winsorizar el mínimo (inferior)
df['pdays'] = winsorize(df['pdays'], limits=[0.05, 0])  # 5% abajo, 0% arriba

# Solo winsorizar el máximo
df['previous'] = winsorize(df['previous'], limits=[0, 0.05]) 

In [0]:
# Histograms
for col in cols_num:
    plt.hist(df[col])
    plt.title(col)
    plt.show()

In [0]:
df.isna().sum()
df.dropna(inplace=True)

In [0]:
# Base line model (decision tree, random forest, logistic regression, xgboost)
#%pip install xgboost

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import make_column_selector as selector
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.metrics import precision_recall_curve, average_precision_score

X = df.drop('y', axis=1)
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'Decision Tree':        DecisionTreeClassifier(random_state=42),
    'Random Forest':        RandomForestClassifier(random_state=42),
    'Logistic Regression':  LogisticRegression(max_iter=1000, random_state=42),
    'XGBoost':              XGBClassifier(random_state=42, eval_metric='logloss')
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = classification_report(y_test, y_pred, output_dict=True)
    print(f"\n{name}:\n", classification_report(y_test, y_pred))
